# AE3: preparación y análisis descriptivo de salarios

## Necesidad y beneficiario

Este ejercicio académico plantea una necesidad hipotética del equipo de talento humano de una empresa que contrata perfiles de datos y tecnología: comparar los salarios registrados en USD por nivel de experiencia, modalidad de trabajo remoto y tamaño de empresa. No se parte de una organización real ni de un problema comprobado; tampoco se afirma que los registros representen todo el mercado.

## Preguntas de análisis

1. ¿Cómo se comparan los salarios en USD (`salary_in_usd`) por nivel de experiencia, modalidad de trabajo remoto y tamaño de empresa?
2. ¿Cuántos registros hay por año de referencia (`work_year`), nivel de experiencia y tipo de empleo?
3. ¿Qué cargos tienen más registros y cuál es su salario mediano en USD?
4. ¿Cómo cambia el salario mediano en USD entre los años cubiertos?

## Variables y resultados esperados

| Pregunta | Variables | Resultado |
|---|---|---|
| Principal | `salary_in_usd`, `experience_level`, `remote_ratio`, `company_size` | Conteo, mediana, promedio y cuartiles del salario por combinación de grupos. |
| Apoyo 1 | `work_year`, `experience_level`, `employment_type` | Conteo por año de referencia, experiencia y tipo de empleo. |
| Apoyo 2 | `job_title`, `salary_in_usd` | Conteo y mediana en USD por cargo. |
| Apoyo 3 | `work_year`, `salary_in_usd` | Conteo, mediana anual y variación frente al año anterior. |

El campo disponible es `work_year`, no una fecha de pago exacta; por eso se usa “año de referencia” y no se atribuye una fecha de pago. El archivo heredado de AE2 es la fuente y se conserva una copia sin modificar. Todas las comparaciones salariales usan `salary_in_usd`; `salary` se mantiene en su moneda original.

## 1. Carga reproducible

Se localiza la raíz del proyecto desde la carpeta de trabajo, se importa el flujo implementado en `procesar_dataset.py` y se carga el CSV original de AE2. La copia de referencia no se altera.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "Entrega_3" / "procesar_dataset.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "src" / "Entrega_3" / "procesar_dataset.py").is_file():
    raise FileNotFoundError("No se encontró src/Entrega_3/procesar_dataset.py desde la carpeta actual.")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
from IPython.display import display
from Entrega_3.procesar_dataset import (
    RAW_PATH,
    RESULTS_DIR,
    PREPARED_PATH,
    cargar_y_transformar,
    describir_cambios,
    crear_tablas_resultados,
    exportar_resultados,
)

df_original, df_preparado = cargar_y_transformar(RAW_PATH)
print(f"Fuente original: {RAW_PATH.relative_to(ROOT)}")
print(f"Filas originales: {len(df_original):,}; columnas originales: {len(df_original.columns)}")
display(df_original.head())

Fuente original: src\Entrega_3\datos\demo-salaries-original.csv
Filas originales: 3,755; columnas originales: 11


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2023,SE,FT,Principal Data Scientist,80000,EUR,85847,ES,100,ES,L
1,2023,MI,CT,ML Engineer,30000,USD,30000,US,100,US,S
2,2023,MI,CT,ML Engineer,25500,USD,25500,US,100,US,S
3,2023,SE,FT,Data Scientist,175000,USD,175000,CA,100,CA,M
4,2023,SE,FT,Data Scientist,120000,USD,120000,CA,100,CA,M


## 2. Transformación y controles de calidad

Los campos numéricos de año, salario original, salario en USD y porcentaje remoto se convierten a enteros anulables. Se normalizan las categorías con etiquetas legibles y se agrega la modalidad laboral desde `remote_ratio`. El código falla explícitamente ante columnas ausentes, códigos no contemplados, valores numéricos no enteros o salarios USD negativos; no elimina filas ni reemplaza faltantes. Se comparan filas, columnas, duplicados exactos y nulos en ambas versiones. El conteo de duplicados corresponde a las ocurrencias posteriores a la primera fila de cada patrón.

In [2]:
cambios = describir_cambios(df_original, df_preparado)
display(cambios)

comparacion_tipos = pd.DataFrame({
    "tipo_original": df_original.dtypes.astype(str),
    "tipo_preparado": df_preparado.dtypes.astype(str),
    "nulos_originales": df_original.isna().sum(),
    "nulos_preparados": df_preparado.isna().sum(),
})
display(comparacion_tipos)
print("Categorías normalizadas y modalidad derivada:")
for columna in ["experience_level", "employment_type", "company_size", "remote_work_mode"]:
    print(f"- {columna}: {sorted(df_preparado[columna].dropna().unique().tolist())}")
print("\nVista del DataFrame preparado:")
display(df_preparado.head())

,etapa,filas,columnas,duplicados_exactos,valores_nulos
0,Original,3755,11,1171,0
1,Preparado,3755,12,1171,0


,tipo_original,tipo_preparado,nulos_originales,nulos_preparados
company_location,object,object,0.0,0
company_size,object,object,0.0,0
employee_residence,object,object,0.0,0
employment_type,object,object,0.0,0
experience_level,object,object,0.0,0
job_title,object,object,0.0,0
remote_ratio,int64,Int64,0.0,0
remote_work_mode,NaN,string,NaN,0
salary,int64,Int64,0.0,0
salary_currency,object,object,0.0,0


Categorías normalizadas y modalidad derivada:
- experience_level: ['Ejecutivo', 'Inicial', 'Intermedio', 'Senior']
- employment_type: ['Contrato', 'Freelance', 'Tiempo completo', 'Tiempo parcial']
- company_size: ['Grande', 'Mediana', 'Pequeña']
- remote_work_mode: ['Híbrido', 'Presencial', 'Remoto']

Vista del DataFrame preparado:


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,remote_work_mode
0,2023,Senior,Tiempo completo,Principal Data Scientist,80000,EUR,85847,ES,100,ES,Grande,Remoto
1,2023,Intermedio,Contrato,ML Engineer,30000,USD,30000,US,100,US,Pequeña,Remoto
2,2023,Intermedio,Contrato,ML Engineer,25500,USD,25500,US,100,US,Pequeña,Remoto
3,2023,Senior,Tiempo completo,Data Scientist,175000,USD,175000,CA,100,CA,Mediana,Remoto
4,2023,Senior,Tiempo completo,Data Scientist,120000,USD,120000,CA,100,CA,Mediana,Remoto


### Diccionario de variables utilizado

Las etiquetas reemplazan códigos crípticos de categorías; se conservan las variables numéricas y las variables de origen restantes. `remote_work_mode` es derivada: 0 = presencial, 50 = híbrido y 100 = remoto. No se agrega una fecha porque el dataset solo proporciona el año.

In [3]:
significados = {
    "work_year": ("Año de referencia registrado en el dataset", "año"),
    "experience_level": ("Nivel de experiencia (Inicial, Intermedio, Senior, Ejecutivo)", "categoría"),
    "employment_type": ("Tipo de empleo (Tiempo completo, Tiempo parcial, Contrato, Freelance)", "categoría"),
    "job_title": ("Cargo reportado", "texto"),
    "salary": ("Salario en la moneda de salary_currency", "unidad de salary_currency; periodicidad no especificada"),
    "salary_currency": ("Código de moneda del salario original", "código de moneda"),
    "salary_in_usd": ("Salario convertido a dólares estadounidenses en el dataset", "USD; periodicidad no especificada"),
    "employee_residence": ("Código de país de residencia del empleado", "código de país"),
    "remote_ratio": ("Porcentaje remoto reportado", "% (0, 50 o 100)"),
    "company_location": ("Código de país de la empresa", "código de país"),
    "company_size": ("Tamaño de empresa (Pequeña, Mediana, Grande)", "categoría"),
    "remote_work_mode": ("Modalidad derivada de remote_ratio", "categoría"),
}
diccionario = pd.DataFrame([
    {
        "variable": nombre,
        "tipo": str(df_preparado[nombre].dtype),
        "significado": significados[nombre][0],
        "unidad_o_regla": significados[nombre][1],
    }
    for nombre in df_preparado.columns
])
display(diccionario)

,variable,tipo,significado,unidad_o_regla
0,work_year,Int64,Año de referencia registrado en el dataset,año
1,experience_level,object,"Nivel de experiencia (Inicial, Intermedio, Sen...",categoría
2,employment_type,object,"Tipo de empleo (Tiempo completo, Tiempo parcia...",categoría
3,job_title,object,Cargo reportado,texto
4,salary,Int64,Salario en la moneda de salary_currency,unidad de salary_currency; periodicidad no esp...
5,salary_currency,object,Código de moneda del salario original,código de moneda
6,salary_in_usd,Int64,Salario convertido a dólares estadounidenses e...,USD; periodicidad no especificada
7,employee_residence,object,Código de país de residencia del empleado,código de país
8,remote_ratio,Int64,Porcentaje remoto reportado,"% (0, 50 o 100)"
9,company_location,object,Código de país de la empresa,código de país


## 3. Pregunta principal: salarios por experiencia, modalidad y tamaño de empresa

La tabla compara cada combinación observada. `cantidad_registros` contextualiza la mediana; promedio y cuartiles muestran la distribución en USD. Las combinaciones inexistentes no se inventan ni se completan con salarios.

In [4]:
tablas = crear_tablas_resultados(df_preparado)
tabla_principal = tablas["pregunta_principal_salarios_por_grupo"]
display(tabla_principal)

grupo_mayor_mediana = tabla_principal.loc[tabla_principal["salario_usd_mediana"].idxmax()]
grupo_mayor_n = tabla_principal.loc[tabla_principal["cantidad_registros"].idxmax()]
print(
    "En los grupos observados, la mediana más alta corresponde a "
    f"{grupo_mayor_mediana['nivel_experiencia']} / {grupo_mayor_mediana['modalidad_trabajo']} / "
    f"{grupo_mayor_mediana['tamano_empresa']}: USD {grupo_mayor_mediana['salario_usd_mediana']:,.0f} "
    f"(n={int(grupo_mayor_mediana['cantidad_registros'])})."
)
print(
    "El grupo con más registros es "
    f"{grupo_mayor_n['nivel_experiencia']} / {grupo_mayor_n['modalidad_trabajo']} / "
    f"{grupo_mayor_n['tamano_empresa']} (n={int(grupo_mayor_n['cantidad_registros'])})."
)
print("Interpretación: las medianas describen solo esta muestra; revise el conteo de cada combinación antes de compararlas.")
print("Limitación: estas comparaciones no controlan país, composición de cargos, inflación ni paridad de poder adquisitivo.")

,nivel_experiencia,modalidad_trabajo,tamano_empresa,cantidad_registros,salario_usd_mediana,salario_usd_promedio,salario_usd_q1,salario_usd_q3
0,Ejecutivo,Híbrido,Grande,5,125976.0,140831.0,79833.00,192037.00
1,Ejecutivo,Híbrido,Mediana,1,130026.0,130026.0,130026.00,130026.00
2,Ejecutivo,Presencial,Grande,4,123923.0,145474.75,100513.25,168884.50
3,Ejecutivo,Presencial,Mediana,52,181080.0,185278.942308,145000.00,212500.00
4,Ejecutivo,Remoto,Grande,4,194333.5,215916.75,152750.25,257500.00
5,Ejecutivo,Remoto,Mediana,42,218500.0,217307.404762,161218.75,255500.00
6,Ejecutivo,Remoto,Pequeña,6,190000.0,196827.166667,131416.50,200000.00
7,Inicial,Híbrido,Grande,30,50000.0,79801.333333,37463.50,95632.00
8,Inicial,Híbrido,Mediana,22,30760.0,42780.227273,22106.00,56826.50
9,Inicial,Híbrido,Pequeña,13,55000.0,63476.923077,28609.00,100000.00


En los grupos observados, la mediana más alta corresponde a Ejecutivo / Remoto / Mediana: USD 218,500 (n=42).
El grupo con más registros es Senior / Presencial / Mediana (n=1269).
Interpretación: las medianas describen solo esta muestra; revise el conteo de cada combinación antes de compararlas.
Limitación: estas comparaciones no controlan país, composición de cargos, inflación ni paridad de poder adquisitivo.


## 4. Pregunta de apoyo 1: registros por año, experiencia y empleo

La tabla incluye las combinaciones presentes en los registros. `work_year` se interpreta como año de referencia del dataset, pues no existe una fecha exacta de pago.

In [5]:
tabla_apoyo_1 = tablas["pregunta_apoyo_1_registros_por_anio_experiencia_empleo"]
display(tabla_apoyo_1)
print(
    f"La tabla contiene {len(tabla_apoyo_1)} combinaciones observadas y "
    f"sus conteos suman {int(tabla_apoyo_1['cantidad_registros'].sum()):,} registros, "
    "el total del dataset."
)
print("Limitación: grupos con pocos registros describen una base reducida y no deben generalizarse.")

,anio_referencia,nivel_experiencia,tipo_empleo,cantidad_registros
0,2020,Ejecutivo,Tiempo completo,3
1,2020,Inicial,Contrato,1
2,2020,Inicial,Tiempo completo,20
3,2020,Inicial,Tiempo parcial,2
4,2020,Intermedio,Tiempo completo,32
5,2020,Senior,Freelance,1
6,2020,Senior,Tiempo completo,17
7,2021,Ejecutivo,Contrato,1
8,2021,Ejecutivo,Tiempo completo,9
9,2021,Inicial,Tiempo completo,50


La tabla contiene 35 combinaciones observadas y sus conteos suman 3,755 registros, el total del dataset.
Limitación: grupos con pocos registros describen una base reducida y no deben generalizarse.


## 5. Pregunta de apoyo 2: cargos con más registros y mediana salarial

El archivo CSV conserva todos los cargos. Se muestra en pantalla el top 20 ordenado por cantidad de registros y, ante empates, por nombre del cargo.

In [6]:
tabla_apoyo_2 = tablas["pregunta_apoyo_2_cargos_por_registros_y_mediana"]
display(tabla_apoyo_2.head(20))
cargo_mas_frecuente = tabla_apoyo_2.iloc[0]
print(
    f"El cargo con más registros es {cargo_mas_frecuente['cargo']} "
    f"(n={int(cargo_mas_frecuente['cantidad_registros'])}); "
    f"su salario mediano es USD {cargo_mas_frecuente['salario_usd_mediano']:,.0f}."
)
print(f"La tabla completa contiene {len(tabla_apoyo_2)} cargos.")
print("La mediana es descriptiva y no representa una banda salarial recomendada para contratar.")

,cargo,cantidad_registros,salario_usd_mediano
0,Data Engineer,1040,135000.0
1,Data Scientist,840,141525.0
2,Data Analyst,612,106020.0
3,Machine Learning Engineer,289,150000.0
4,Analytics Engineer,103,143860.0
5,Data Architect,101,160000.0
6,Research Scientist,82,150000.0
7,Applied Scientist,58,191737.5
8,Data Science Manager,58,183780.0
9,Research Engineer,37,155000.0


El cargo con más registros es Data Engineer (n=1040); su salario mediano es USD 135,000.
La tabla completa contiene 93 cargos.
La mediana es descriptiva y no representa una banda salarial recomendada para contratar.


## 6. Pregunta de apoyo 3: variación de la mediana entre años

Se calcula la diferencia absoluta en USD y la variación porcentual respecto a la mediana del año de referencia inmediatamente anterior. Para el primer año no hay año previo en la muestra, por lo que la variación queda nula.

In [7]:
tabla_apoyo_3 = tablas["pregunta_apoyo_3_salario_mediano_por_anio"]
display(tabla_apoyo_3)
for _, fila in tabla_apoyo_3.iterrows():
    mensaje = f"{int(fila['anio_referencia'])}: mediana USD {fila['salario_usd_mediano']:,.0f}"
    if pd.notna(fila["variacion_mediana_usd_vs_anio_anterior"]):
        mensaje += (
            f"; cambio USD {fila['variacion_mediana_usd_vs_anio_anterior']:+,.0f} "
            f"({fila['variacion_mediana_porcentual_vs_anio_anterior']:+.1f}%)"
        )
    print(mensaje)
print("Las variaciones comparan medianas de registros; no identifican causas ni controlan cambios en la composición de cargos.")

,anio_referencia,cantidad_registros,salario_usd_mediano,variacion_mediana_usd_vs_anio_anterior,variacion_mediana_porcentual_vs_anio_anterior
0,2020,76,73065.0,<NA>,<NA>
1,2021,230,80000.0,6935.0,9.491549
2,2022,1664,131300.0,51300.0,64.125
3,2023,1785,143860.0,12560.0,9.56588


2020: mediana USD 73,065
2021: mediana USD 80,000; cambio USD +6,935 (+9.5%)
2022: mediana USD 131,300; cambio USD +51,300 (+64.1%)
2023: mediana USD 143,860; cambio USD +12,560 (+9.6%)
Las variaciones comparan medianas de registros; no identifican causas ni controlan cambios en la composición de cargos.


## 7. Exportación y comprobación de archivos

Se exportan el DataFrame final, una tabla CSV por pregunta y el resumen `.txt` con dimensiones, columnas y tipos, duplicados, nulos y estadísticas numéricas. El mismo código de análisis utilizado en este notebook es invocable desde la terminal con `python src\Entrega_3\procesar_dataset.py`.

In [8]:
archivos_generados = exportar_resultados(df_preparado, tablas, RESULTS_DIR, PREPARED_PATH)
df_exportado = pd.read_csv(PREPARED_PATH, float_precision="round_trip")
assert len(df_exportado) == len(df_preparado), "El CSV preparado no coincide en filas con el DataFrame del notebook."
assert df_exportado.shape[1] == len(df_preparado.columns), "El CSV preparado no coincide en columnas con el DataFrame del notebook."
pd.testing.assert_frame_equal(df_exportado, df_preparado.reset_index(drop=True), check_dtype=False, check_exact=True)
for nombre, tabla in tablas.items():
    ruta = archivos_generados[nombre]
    tabla_exportada = pd.read_csv(ruta, float_precision="round_trip")
    esperado = tabla.reset_index(drop=True).astype(object)
    observado = tabla_exportada.astype(object)
    esperado = esperado.where(pd.notna(esperado), None)
    observado = observado.where(pd.notna(observado), None)
    pd.testing.assert_frame_equal(observado, esperado, check_dtype=False, check_exact=True)
    assert int(tabla_exportada["cantidad_registros"].sum()) == len(df_preparado)

for nombre, ruta in archivos_generados.items():
    print(f"{nombre}: {ruta.relative_to(ROOT)} ({ruta.stat().st_size:,} bytes)")
print("\nValidación completada: filas del dataset y conteos de las cuatro tablas coinciden.")
print(f"Dataset preparado: {df_exportado.shape[0]:,} filas × {df_exportado.shape[1]} columnas")

dataset_preparado: src\Entrega_3\datos\demo-salaries-preparado.csv (338,001 bytes)
pregunta_principal_salarios_por_grupo: src\Entrega_3\resultados\pregunta_principal_salarios_por_grupo.csv (2,564 bytes)
pregunta_apoyo_1_registros_por_anio_experiencia_empleo: src\Entrega_3\resultados\pregunta_apoyo_1_registros_por_anio_experiencia_empleo.csv (1,154 bytes)
pregunta_apoyo_2_cargos_por_registros_y_mediana: src\Entrega_3\resultados\pregunta_apoyo_2_cargos_por_registros_y_mediana.csv (3,183 bytes)
pregunta_apoyo_3_salario_mediano_por_anio: src\Entrega_3\resultados\pregunta_apoyo_3_salario_mediano_por_anio.csv (298 bytes)
resumen_dataframe: src\Entrega_3\resultados\resumen_dataframe.txt (1,279 bytes)

Validación completada: filas del dataset y conteos de las cuatro tablas coinciden.
Dataset preparado: 3,755 filas × 12 columnas
